In [ ]:
# -*- coding: utf-8 -*-
"""
DB 数据批量打标签脚本（优化终版）
---------------------------------------------------
本版包含：
1) 公费师范 / 师范 冲突裁决：同时命中时，仅保留“公费师范”
2) 师范匹配修正：
   - “非师范”不能命中“师范”
   - “非公费师范”可以命中“师范”，但不能命中“公费师范”
   - “师范学院 / 师范大学 / 师范专科 / 师范高等”不能作为“师范”标签命中依据
   - “师范”标签的提取字段固定为“师范”
3) 新增“户籍”列
   - 多个逻辑都提取出 xx 时，只取长度最小的那个
   - 若长度相同，取最先匹配到的那个
4) 速度优化：
   - 规则 key 预归一化 / 预编译
   - 通配符 regex 仅编译一次
   - 户籍 regex 仅编译一次
   - 行扫描使用 iter_rows，减少重复 cell 访问
   - 文本归一与否定判断尽量只做一次
5) 新增户籍统一输出开关：
   - 开启后，只要“户籍”字段非空，统一写成“有户籍要求”
   - 关闭后，保留原始提取到的户籍内容

依赖：openpyxl
"""

import os
import re
from openpyxl import load_workbook

# =========================
# 配置区（你自己改路径）
# =========================
RULE_PATH = r"规则表/各省份升学路径细分粗略版 copy.xlsx"
INPUT_PATH = r"/Users/bangongyi/Downloads/最新招生计划数据-z x"   # 支持：单个文件 或 目录
OUTPUT_DIR = r"/Users/bangongyi/Desktop/更新数据流程文档/25全国招生计划4月最新"

# 规则表列名
RULE_COL_PROVINCE = "省份"
RULE_COL_LABEL = "细分标签"
RULE_COL_KEYS = "提取字段"

# DB 数据列名
DB_COL_PROVINCE = "生源地"
DB_COL_MAJOR_NAME = "专业名称"
DB_COL_MAJOR_DIR = "专业代码"
DB_COL_MAJOR_REMARK = "专业备注"
DB_COL_SCHOOL = "院校名称"

# 备用字段（最多 5 个）：参与“专业全称”合成
DB_COL_FULLNAME_EXTRA_1 = "批次"
DB_COL_FULLNAME_EXTRA_2 = "专业组备注"
DB_COL_FULLNAME_EXTRA_3 = ""
DB_COL_FULLNAME_EXTRA_4 = ""
DB_COL_FULLNAME_EXTRA_5 = ""

# 写回列
DB_COL_MAJOR_FULLNAME = "专业全称"
OUT_COL_HITS = "原文字段"
OUT_COL_MAJOR_NATURE = "专业性质"
OUT_COL_HUKOU = "户籍地限制"
HUKOU_INSERT_AFTER_COL = "身体健康状态"

# 输出开关
OUTPUT_HITS_COLUMN = False
OUTPUT_MAJOR_FULLNAME_COLUMN = False

# 户籍统一输出开关
# True  = 跑完后，“户籍”字段非空统一写成“1”，为空统一写成“0”
# False = 保留原始提取到的户籍内容
UNIFY_NONEMPTY_HUKOU_TO_FLAG = True

# 特殊与冲突规则
HAINAN_ETHNIC_LABEL = "民族班"
HAINAN_PREP_LABEL = "预科班"
CONFLICT_PUBLIC_NORMAL_LABEL = "公费师范"
CONFLICT_NORMAL_TEACHER_LABEL = "师范"
CONFLICT_GAOXIAO_SPECIAL_LABEL = "高校专项"
CONFLICT_GUOJIA_SPECIAL_LABEL = "国家专项"
CONFLICT_DIFANG_SPECIAL_LABEL = "地方专项"
CONFLICT_ZHONGWAI_LABEL = "中外合作"
FREE_MEDICAL_LABEL = "免费医学"
FREE_MEDICAL_DIRECTIONAL_TOKEN = "定向"
FREE_MEDICAL_INVALID_DIRECTIONAL_FOLLOW = "培养军"
SPECIAL_SEGMENT_BREAK_CHARS = ",，。；;:：()（）【】[]<>《》、/\\|+-"
NEG_CONTEXT_GUOJIA_SPECIAL_WITH_HAN = re.compile(
    r"含[^,，。；;:：()（）【】\[\]<>《》、/\\|+\-]{0,5}国家专项(?:计划)?"
)
NEG_CONTEXT_DIFANG_SPECIAL_WITH_GUOJIA = re.compile(
    r"国家及地方专项"
)
NEG_CONTEXT_GAOXIAO_SPECIAL_LIAONING = re.compile(
    r"(?:符合高水平运动队,高校专项(?:计划)?资格|含高校专项)"
)
NEG_CONTEXT_PREP_SPECIAL_INVALID = re.compile(
    r"区域教育均衡专项计划,?省属高校少数民族预科"
)
NEG_CONTEXT_SCHOOL_ENTERPRISE_UNION_INVALID = re.compile(
    r"校企联合实验室"
)
NEG_CONTEXT_FREE_MEDICAL_YI_INVALID = re.compile(
    r"(?:动物医学|兽医)"
)
NEG_CONTEXT_FREE_MEDICAL_DIRECTIONAL_INVALID = re.compile(
    r"非西藏生源定向西藏就业"
)

# 公费否定语境
NEG_CONTEXT_GONGFEI = re.compile(
    r"(不招收|不招|不录取|不予录取|不接收|不收|禁止|不得|不允许|非|不包含)[^,，。；;:：()（）【】\[\]\s]{0,0}(公费师范生|公费师范|公费农科生|公费医学)"
)

# “公费师范”的否定语境，同时也应判定为不属于“师范”
NEG_CONTEXT_PUBLIC_NORMAL_FOR_SHIFAN = re.compile(
    r"(?:"
    r"(?:公费师范(?:生)?)[^,，。；;:：()（）【】\[\]<>《》、/\\|+\-\s]{0,5}(?:除外|不含|不包括)"
    r")|(?:"
    r"(?:除外|不含|不包括|不招|不招收|不允许|不录取|不予录取|不接收|不收|禁止|不得|非|不包含)"
    r"[^,，。；;:：()（）【】\[\]<>《》、/\\|+\-\s]{0,5}公费师范(?:生)?"
    r")"
)
# 中外合作否定语境：专门用于剔除“中外合作”标签
# 兼容“国际班”作为中外合作的提取字段别名：
# 例如“不含国际班”“国际班除外”“不包括国际班项目”都应剔除“中外合作”标签
NEG_CONTEXT_ZHONGWAI = re.compile(
    r"(?:"
    r"(?:中外合作(?:办学|专业|项目|方向)?|国际班(?:项目|方向)?)[^,，。；;:：()（）【】\[\]\s]{0,1}(?:除外|不含|不包括)"
    r")|(?:"
    r"(?:除外|不含|不包括)[^,，。；;:：()（）【】\[\]\s]{0,1}(?:中外合作(?:办学|专业|项目|方向)?|国际班(?:项目|方向)?)"
    r")"
)

# =========================
# 文本归一
# =========================
_symbol_map = {
    "（": "(", "）": ")",
    "【": "[", "】": "]",
    "，": ",", "、": ",",
    "；": ";", "：": ":",
    "－": "-", "—": "-", "–": "-",
    "＋": "+", "／": "/", "．": ".",
    "\u200b": "", "\xa0": "", "\u3000": "",
    "“": "", "”": "", "‘": "", "’": "",
    "　": " ", "\t": " ",
}
_symbol_re = re.compile("|".join(map(re.escape, _symbol_map.keys())))
_space_re = re.compile(r"\s+")


def unify_symbols(s: str) -> str:
    if not s:
        return ""
    return _symbol_re.sub(lambda m: _symbol_map[m.group(0)], str(s))


def remove_all_spaces(s: str) -> str:
    if not s:
        return ""
    s = str(s).replace("\u200b", "").replace("\xa0", "").replace("\u3000", "")
    return _space_re.sub("", s)


def norm_text_for_match(s: str) -> str:
    return remove_all_spaces(unify_symbols(s))


def norm_text_for_dedup(s: str) -> str:
    return norm_text_for_match(s)


# =========================
# 工具函数
# =========================
def header_to_col(ws):
    m = {}
    for c in range(1, ws.max_column + 1):
        v = ws.cell(row=1, column=c).value
        if v is not None:
            m[str(v).strip()] = c
    return m


def ensure_col(ws, header_map, name):
    if name in header_map:
        return header_map[name]
    new_col = ws.max_column + 1
    ws.cell(row=1, column=new_col).value = name
    header_map[name] = new_col
    return new_col


def refresh_header_map(ws, header_map):
    header_map.clear()
    for c in range(1, ws.max_column + 1):
        v = ws.cell(row=1, column=c).value
        if v is not None:
            header_map[str(v).strip()] = c
    return header_map


def ensure_col_after(ws, header_map, name, after_name):
    if name in header_map:
        return header_map[name]

    after_name = get_str(after_name)
    if not after_name or after_name not in header_map:
        return ensure_col(ws, header_map, name)

    insert_at = header_map[after_name] + 1
    ws.insert_cols(insert_at, 1)
    ws.cell(row=1, column=insert_at).value = name
    refresh_header_map(ws, header_map)
    return header_map[name]


def get_str(v):
    return "" if v is None else str(v).strip()


def split_multikey_cell(cell_text: str):
    if not cell_text:
        return []
    s = str(cell_text).strip()
    if not s:
        return []
    keys = []
    for line in re.split(r"[\r\n]+", s):
        line = line.strip()
        if not line:
            continue
        keys.extend([p.strip() for p in line.split("/") if p.strip()])
    return keys


def parse_key_expr(key: str):
    key = key.strip()
    if not key:
        return None
    if "+" in key:
        parts = [p.strip() for p in key.split("+") if p.strip()]
        return ("AND", parts, key) if parts else None
    return ("SINGLE", [key], key)


def is_wildcard_key(key: str):
    return ("XXX" in key) or ("xxx" in key) or ("X" in key) or ("x" in key)


def compile_wildcard_regex(key: str):
    esc = re.escape(key)
    esc = esc.replace(re.escape("XXX"), r"(.+?)")
    esc = esc.replace(re.escape("xxx"), r"(.+?)")
    esc = esc.replace(re.escape("X"), r"(.+?)")
    esc = esc.replace(re.escape("x"), r"(.+?)")
    return re.compile(esc)


def token_sort_key(item):
    return (len(item["match_raw"]), sum(len(p) for p in item["match_parts"]))


# =========================
# 匹配逻辑
# =========================
NEG_PREFIX_GONGFEI = {"非", "不"}
NEG_PREFIX_SHIFAN = {"非"}
SHIFAN_EXCLUDE_SUFFIXES = ("学院", "大学", "专科", "高等","学校","院校")
NEG_CONTEXT_SHIFAN = re.compile(
    r"(?:"
    r"(?:师范(?:专业|类)?)[^,，。；;:：()（）【】\[\]<>《》、/\\|+\-\s]{0,5}(?:除外|不含|不包括|不包含)"
    r")|(?:"
    r"(?:除外|不含|不包括|不包含|不招|不招收)[^,，。；;:：()（）【】\[\]<>《》、/\\|+\-\s]{0,5}师范(?:专业|类)?"
    r")"
)


def is_index_in_regex_span(pattern, text: str, idx: int) -> bool:
    for match in pattern.finditer(text):
        if match.start() <= idx < match.end():
            return True
    return False


def has_valid_free_medical_directional(text: str) -> bool:
    start = 0
    token = FREE_MEDICAL_DIRECTIONAL_TOKEN
    token_len = len(token)
    invalid_follow = FREE_MEDICAL_INVALID_DIRECTIONAL_FOLLOW

    while True:
        idx = text.find(token, start)
        if idx == -1:
            return False
        suffix = text[idx + token_len: idx + token_len + len(invalid_follow)]
        invalid_follow_hit = suffix.startswith(invalid_follow)
        invalid_special_ctx_hit = is_index_in_regex_span(
            NEG_CONTEXT_FREE_MEDICAL_DIRECTIONAL_INVALID,
            text,
            idx,
        )
        if not invalid_follow_hit and not invalid_special_ctx_hit:
            return True
        start = idx + 1


def has_invalid_han_prefix_for_guojia_special(text: str, idx: int) -> bool:
    if is_index_in_regex_span(NEG_CONTEXT_GUOJIA_SPECIAL_WITH_HAN, text, idx):
        return True

    seg_start = idx
    while seg_start > 0 and text[seg_start - 1] not in SPECIAL_SEGMENT_BREAK_CHARS:
        seg_start -= 1

    prefix = text[seg_start:idx]
    return "含" in prefix


def has_invalid_guojia_prefix_for_difang_special(text: str, idx: int) -> bool:
    return is_index_in_regex_span(NEG_CONTEXT_DIFANG_SPECIAL_WITH_GUOJIA, text, idx)


def has_invalid_liaoning_prefix_for_gaoxiao_special(text: str, idx: int) -> bool:
    return is_index_in_regex_span(NEG_CONTEXT_GAOXIAO_SPECIAL_LIAONING, text, idx)


def has_invalid_special_context_for_prep(text: str, idx: int) -> bool:
    return is_index_in_regex_span(NEG_CONTEXT_PREP_SPECIAL_INVALID, text, idx)


def has_invalid_special_context_for_school_enterprise_union(text: str, idx: int) -> bool:
    return is_index_in_regex_span(NEG_CONTEXT_SCHOOL_ENTERPRISE_UNION_INVALID, text, idx)


def has_valid_free_medical_yi(text: str) -> bool:
    start = 0
    literal = "医"

    while True:
        idx = text.find(literal, start)
        if idx == -1:
            return False
        if not is_index_in_regex_span(NEG_CONTEXT_FREE_MEDICAL_YI_INVALID, text, idx):
            return True
        start = idx + 1


def has_non_negated_literal(text: str, literal: str) -> bool:
    """
    规则：
    1) literal == “师范”
       - 前一字为“非”时，不算命中
       - 后面若紧跟 学院/大学/专科/高等，不算命中
       - 若出现“不含师范 / 不包括师范 / 师范除外 / 师范专业除外 / 不招师范”等否定语境，不算命中
       - “非公费师范”以及“公费师范”的否定语境，也不属于“师范”
    2) 含“公费”的 literal
       - 前一字为“非/不”时，不算命中
    3) literal 含“国家专项”
       - 若前面同一符号段里出现“含”，则该次命中无效，且不能跨越符号判断
    4) literal 含“地方专项”
       - 若该次命中位于“国家及地方专项”中，则该次命中无效
    5) literal 含“高校专项”
       - 若该次命中位于“辽宁省高校专项计划”中，则该次命中无效
    6) literal 含“预科”
       - 若该次命中位于“区域教育均衡专项计划、省属高校少数民族预科”中，则该次命中无效
    7) literal 含“校企联合”
       - 若该次命中位于“校企联合实验室”中，则该次命中无效
    8) 其他 literal：普通包含
    """
    if literal == CONFLICT_NORMAL_TEACHER_LABEL:
        if NEG_CONTEXT_SHIFAN.search(text):
            return False

        start = 0
        lit_len = len(literal)
        text_len = len(text)
        while True:
            idx = text.find(literal, start)
            if idx == -1:
                return False
            prev_ok = (idx == 0 or text[idx - 1] not in NEG_PREFIX_SHIFAN)
            local_public_normal_negated = is_index_in_regex_span(NEG_CONTEXT_PUBLIC_NORMAL_FOR_SHIFAN, text, idx)
            if prev_ok and not local_public_normal_negated:
                suffix = text[idx + lit_len: idx + lit_len + 2] if idx + lit_len < text_len else ""
                if not any(suffix.startswith(x) for x in SHIFAN_EXCLUDE_SUFFIXES):
                    return True
            start = idx + 1

    if "公费" in literal:
        start = 0
        while True:
            idx = text.find(literal, start)
            if idx == -1:
                return False
            if idx == 0 or text[idx - 1] not in NEG_PREFIX_GONGFEI:
                return True
            start = idx + 1

    if CONFLICT_GUOJIA_SPECIAL_LABEL in literal:
        start = 0
        while True:
            idx = text.find(literal, start)
            if idx == -1:
                return False
            if not has_invalid_han_prefix_for_guojia_special(text, idx):
                return True
            start = idx + 1

    if CONFLICT_DIFANG_SPECIAL_LABEL in literal:
        start = 0
        while True:
            idx = text.find(literal, start)
            if idx == -1:
                return False
            if not has_invalid_guojia_prefix_for_difang_special(text, idx):
                return True
            start = idx + 1

    if CONFLICT_GAOXIAO_SPECIAL_LABEL in literal:
        start = 0
        while True:
            idx = text.find(literal, start)
            if idx == -1:
                return False
            if not has_invalid_liaoning_prefix_for_gaoxiao_special(text, idx):
                return True
            start = idx + 1

    if "预科" in literal:
        start = 0
        while True:
            idx = text.find(literal, start)
            if idx == -1:
                return False
            if not has_invalid_special_context_for_prep(text, idx):
                return True
            start = idx + 1

    if "校企联合" in literal:
        start = 0
        while True:
            idx = text.find(literal, start)
            if idx == -1:
                return False
            if not has_invalid_special_context_for_school_enterprise_union(text, idx):
                return True
            start = idx + 1

    return literal in text


def match_item(text: str, item: dict):
    regex = item.get("regex")
    if regex is not None:
        m = regex.search(text)
        if not m:
            return False, None
        return True, m.group(0)

    kind = item["kind"]
    parts = item["match_parts"]
    raw = item["raw"]
    label = item.get("label", "")

    if kind == "SINGLE":
        if label == FREE_MEDICAL_LABEL and parts[0] == FREE_MEDICAL_DIRECTIONAL_TOKEN:
            ok = has_valid_free_medical_directional(text)
        elif label == FREE_MEDICAL_LABEL and parts[0] == "医":
            ok = has_valid_free_medical_yi(text)
        else:
            ok = has_non_negated_literal(text, parts[0])
        return ok, (raw if ok else None)

    if kind == "AND":
        for p in parts:
            if label == FREE_MEDICAL_LABEL and p == FREE_MEDICAL_DIRECTIONAL_TOKEN:
                if not has_valid_free_medical_directional(text):
                    return False, None
            elif label == FREE_MEDICAL_LABEL and p == "医":
                if not has_valid_free_medical_yi(text):
                    return False, None
            else:
                if not has_non_negated_literal(text, p):
                    return False, None
        return True, raw

    return False, None


# =========================
# 专业全称合成
# =========================
def build_major_fullname(parts):
    out_parts = []
    joined_norm = ""
    for p in parts:
        p = get_str(p)
        if not p:
            continue
        p_show = unify_symbols(p).strip()
        p_norm = norm_text_for_dedup(p_show)
        if not p_norm:
            continue
        if joined_norm and (p_norm in joined_norm):
            continue
        out_parts.append(p_show)
        joined_norm += p_norm
    return "".join(out_parts)


# =========================
# 户籍提取
# =========================
_HUKOU_SUFFIXES_TO_STRIP = ("少数民族", "定向", "全省")
_HUKOU_TRIM_CHARS = ",;:()[]，；：（）【】"

HUKOU_GARBAGE_SNIPPETS = (
    "户籍考生免学费",
    "面向就业",
    "入警就业",
    "机关就业",
    "部门就业",
    "英语语种",
    "语考生",
    "科目考生",
    "测试入围",
    "边防军人子女",
    "资格审核",
    "民语类",
    "综合评价",
    "信息学学科领域具有突出才能和表现的",
)
HUKOU_GARBAGE_REGEXES = [
    re.compile(r"只招(?:收)?[^(),，。；;:：]*?语种考生"),
    re.compile(r"只招(?:收)?[^(),，。；;:：]*?语考生"),
    re.compile(r"只招(?:收)?[^(),，。；;:：]*?科目考生"),
    re.compile(r"只招(?:收)?[^(),，。；;:：]*?志愿的考生"),
    re.compile(r"只招(?:收)?[^(),，。；;:：]*?族考生"),
    re.compile(r"只招(?:收)?[^(),，。；;:：]*?文考生"),
    re.compile(r"只招(?:收)?[^(),，。；;:：]*?为主考生"),
    re.compile(r"只招(?:收)?[^(),，。；;:：]*?语的考生"),
    re.compile(r"只招(?:收)?[^(),，。；;:：]*?格的考生"),
    re.compile(r"只招(?:收)?[^(),，。；;:：]*?件的考生"),
    re.compile(r"只招(?:收)?[^(),，。；;:：]*?格考生"),
    re.compile(r"只招(?:收)?[^(),，。；;:：]*?线的考生"),
]

HUKOU_PATTERNS = [
    re.compile(r"\(面向(?P<region>[^()]*?市)\)"),
    re.compile(r"只录取有专业志愿的(?P<region>.+?)户籍"),
    re.compile(r"只招收户籍和学籍均在(?P<region>.+?)考生"),
    re.compile(r"考生本人及监护人须具有(?P<region>.+?)户籍"),
    re.compile(r"招收符合我市普通高考报名条件的(?P<region>.+?)户籍"),
    re.compile(r"招(?P<region>.+?)户籍应届高中毕业生"),
    re.compile(r"面向(?P<region>[^(),，。；;:：]*?)(?=,?招[^()，。；;:：]*?语种考生)"),
    re.compile(r"面向(?P<region>.+?)定向招生"),
    re.compile(r"面向(?P<region>[^(),，。；;:：]*?)户籍"),
    re.compile(r"只招收(?P<region>[^(),，。；;:：]*?)考生"),
    re.compile(r"(?<![不非])招收(?P<region>[^(),，。；;:：]*?)考生"),
    re.compile(r"只录取有专业志愿的(?P<region>[^(),，。；;:：]*?)户籍"),
    re.compile(r"只招收具有(?P<region>[^(),，。；;:：]*?)的考生"),
    re.compile(r"要求(?P<region>[^(),.;:]*?常住户口)(?=[,;:()]|$)"),
    re.compile(r"只招(?!收户籍和学籍均在)(?P<region>[^(),，。；;:：]*?)户籍"),
    re.compile(r"限(?P<region>[^(),，。；;:：]*?)户籍"),
    re.compile(r"面向(?P<region>[^(),，。；;:：]*?)(?:男生|女生)"),
    re.compile(r"面向(?P<region>[^,，。；;:：]*?)(?:生源)"),
    re.compile(r"面向(?P<region>[^(),，。；;:：]*?)(?:招生)"),
    re.compile(r"(?P<region>[^(),，。；;:：]*?)走读"),
    re.compile(r"只招收户籍为(?P<region>[^(),，。；;:：]*?)的考生"),
    re.compile(r"生源范围:(?P<region>[^();]*?)(?=;在|;|\)|$)"),
]

HUKOU_FALLBACK_PATTERNS = [
    re.compile(r"只招收户籍等符合报考条件"),
    re.compile(r"本人具有当地连续3年以上户籍"),
    re.compile(r"招全省农村考生"),
    re.compile(r"具有连续3年以上农村户籍"),
    re.compile(r"具有定向培养市县户籍"),
    re.compile(r"三区三州深度贫困脱贫县"),
]


def clean_hukou_region(region: str) -> str:
    s = norm_text_for_match(region)
    if not s:
        return ""

    if s.startswith("收"):
        s = s[1:]

    if s.endswith("户籍"):
        s = s[:-2]

    changed = True
    while changed and s:
        changed = False
        for suffix in _HUKOU_SUFFIXES_TO_STRIP:
            if s.endswith(suffix):
                s = s[:-len(suffix)]
                changed = True

    s = s.replace("以下", "")

    return s.strip(_HUKOU_TRIM_CHARS)


def extract_hukou_shortest(text: str):
    if not text:
        return None

    best = None
    best_len = None
    best_pos = None

    for pattern in HUKOU_PATTERNS:
        for m in pattern.finditer(text):
            full_hit = m.group(0)

            if any(bad in full_hit for bad in HUKOU_GARBAGE_SNIPPETS):
                continue

            if any(rgx.search(full_hit) for rgx in HUKOU_GARBAGE_REGEXES):
                continue

            if full_hit.startswith("面向") and "就业" in full_hit:
                continue

            if "语种考生" in full_hit or "语考生" in full_hit or "科目考生" in full_hit or "族考生" in full_hit or "文考生" in full_hit or "为主考生" in full_hit or "语的考生" in full_hit or "件的考生" in full_hit or "格的考生" in full_hit or "格考生" in full_hit or "线的考生" in full_hit:
                continue

            region = clean_hukou_region(m.group("region"))
            if not region:
                continue

            if "走读" in region:
                continue
            if "就业" in region:
                continue
            if "语种" in region or "语考生" in full_hit:
                continue
            if "科目" in region or "科目考生" in full_hit:
                continue
            if "志愿" in region:
                continue

            cur_len = len(region)
            cur_pos = m.start()

            if best is None or cur_len < best_len or (cur_len == best_len and cur_pos < best_pos):
                best = region
                best_len = cur_len
                best_pos = cur_pos

    return best


# =========================
# 规则构建
# =========================
def build_rules(rule_ws):
    h = header_to_col(rule_ws)
    for need in [RULE_COL_PROVINCE, RULE_COL_LABEL, RULE_COL_KEYS]:
        if need not in h:
            raise ValueError(f"规则表缺少列：{need}")

    col_p = h[RULE_COL_PROVINCE]
    col_l = h[RULE_COL_LABEL]
    col_k = h[RULE_COL_KEYS]

    prov_map = {}
    label_keycount = {}

    for r in range(2, rule_ws.max_row + 1):
        prov = get_str(rule_ws.cell(r, col_p).value)
        label = get_str(rule_ws.cell(r, col_l).value)
        keys_cell = get_str(rule_ws.cell(r, col_k).value)

        if not prov or not label:
            continue

        # 师范标签提取字段固定为其本身
        if label == CONFLICT_NORMAL_TEACHER_LABEL:
            keys = [CONFLICT_NORMAL_TEACHER_LABEL]
        else:
            keys = split_multikey_cell(keys_cell)
            if not keys:
                keys = [label]

        seen = set()
        keys2 = []
        for k in keys:
            k = k.strip()
            if not k or k in seen:
                continue
            keys2.append(k)
            seen.add(k)

        label_keycount[label] = max(label_keycount.get(label, 0), len(keys2))

        items = []
        for k in keys2:
            parsed = parse_key_expr(k)
            if not parsed:
                continue
            kind, parts, raw = parsed
            match_raw = norm_text_for_match(raw)
            match_parts = [norm_text_for_match(p) for p in parts]
            regex = compile_wildcard_regex(match_raw) if is_wildcard_key(match_raw) else None
            items.append({
                "kind": kind,
                "raw": raw,
                "match_raw": match_raw,
                "match_parts": match_parts,
                "regex": regex,
                "label": label,
            })

        items.sort(key=token_sort_key, reverse=True)
        prov_map.setdefault(prov, []).append({"label": label, "items": items})

    all_labels = sorted(label_keycount.keys())
    return prov_map, all_labels, label_keycount


# =========================
# 冲突裁决
# =========================
def resolve_label_conflicts(prov_val: str, matched_labels, hit_keys):
    if not matched_labels:
        return matched_labels, hit_keys

    pairs = list(zip(matched_labels, hit_keys))
    label_set = set(matched_labels)

    if prov_val == "海南" and HAINAN_ETHNIC_LABEL in label_set and HAINAN_PREP_LABEL in label_set:
        pairs = [(lbl, hk) for lbl, hk in pairs if lbl != HAINAN_PREP_LABEL]
        label_set = {lbl for lbl, _ in pairs}

    if CONFLICT_PUBLIC_NORMAL_LABEL in label_set and CONFLICT_NORMAL_TEACHER_LABEL in label_set:
        pairs = [(lbl, hk) for lbl, hk in pairs if lbl != CONFLICT_NORMAL_TEACHER_LABEL]
        label_set = {lbl for lbl, _ in pairs}

    if CONFLICT_GAOXIAO_SPECIAL_LABEL in label_set and CONFLICT_ZHONGWAI_LABEL in label_set:
        pairs = [(lbl, hk) for lbl, hk in pairs if lbl != CONFLICT_ZHONGWAI_LABEL]
        label_set = {lbl for lbl, _ in pairs}

    if CONFLICT_GUOJIA_SPECIAL_LABEL in label_set and CONFLICT_ZHONGWAI_LABEL in label_set:
        pairs = [(lbl, hk) for lbl, hk in pairs if lbl != CONFLICT_ZHONGWAI_LABEL]
        label_set = {lbl for lbl, _ in pairs}

    if CONFLICT_DIFANG_SPECIAL_LABEL in label_set and CONFLICT_ZHONGWAI_LABEL in label_set:
        pairs = [(lbl, hk) for lbl, hk in pairs if lbl != CONFLICT_ZHONGWAI_LABEL]
        label_set = {lbl for lbl, _ in pairs}

    if not pairs:
        return [], []
    return [x[0] for x in pairs], [x[1] for x in pairs]


def remove_negative_zhongwai_label(remark: str, matched_labels, hit_keys):
    """
    专门针对“中外合作”标签：
    如果备注里出现“不含中外合作办学 / 中外合作除外 / 不包括中外合作项目”
    以及“不含国际班 / 国际班除外 / 不包括国际班项目”等否定语境，
    就把“中外合作”标签从结果中剔除。
    """
    if not matched_labels:
        return matched_labels, hit_keys

    remark_text = norm_text_for_match(remark)
    if not remark_text:
        return matched_labels, hit_keys

    if "中外合作" not in matched_labels:
        return matched_labels, hit_keys

    if not NEG_CONTEXT_ZHONGWAI.search(remark_text):
        return matched_labels, hit_keys

    pairs = [(lbl, hk) for lbl, hk in zip(matched_labels, hit_keys) if lbl != "中外合作"]
    if not pairs:
        return [], []
    return [x[0] for x in pairs], [x[1] for x in pairs]


# =========================
# 处理单个文件
# =========================
def process_one_file(filepath, prov_rules, all_labels, label_keycount):
    keep_vba = filepath.lower().endswith(".xlsm")
    wb = load_workbook(filepath, keep_vba=keep_vba)
    ws = wb.active
    h = header_to_col(ws)

    need_cols = [
        DB_COL_PROVINCE,
        DB_COL_MAJOR_NAME,
        DB_COL_MAJOR_DIR,
        DB_COL_MAJOR_REMARK,
        DB_COL_SCHOOL,
    ]

    extra_cols = [
        DB_COL_FULLNAME_EXTRA_1,
        DB_COL_FULLNAME_EXTRA_2,
        DB_COL_FULLNAME_EXTRA_3,
        DB_COL_FULLNAME_EXTRA_4,
        DB_COL_FULLNAME_EXTRA_5,
    ]
    extra_need = [c for c in extra_cols if isinstance(c, str) and c.strip()]
    need_cols.extend(extra_need)

    miss = [c for c in need_cols if c not in h]
    if miss:
        raise ValueError(f"{os.path.basename(filepath)} 缺少列：{miss}")

    col_prov = h[DB_COL_PROVINCE]
    col_major = h[DB_COL_MAJOR_NAME]
    col_dir = h[DB_COL_MAJOR_DIR]
    col_rem = h[DB_COL_MAJOR_REMARK]
    col_school = h[DB_COL_SCHOOL]
    extra_col_idx = [h[c] for c in extra_need]

    col_full = ensure_col(ws, h, DB_COL_MAJOR_FULLNAME) if OUTPUT_MAJOR_FULLNAME_COLUMN else None

    row_cache = [None] * (ws.max_row + 1)

    idx_prov = col_prov - 1
    idx_major = col_major - 1
    idx_dir = col_dir - 1
    idx_rem = col_rem - 1
    idx_school = col_school - 1
    extra_idx0 = [c - 1 for c in extra_col_idx]

    for r, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), start=2):
        prov_val = get_str(row[idx_prov].value)
        major_name = get_str(row[idx_major].value)
        direction = get_str(row[idx_dir].value)
        remark = get_str(row[idx_rem].value)
        _school = get_str(row[idx_school].value)
        extra_vals = [get_str(row[i].value) for i in extra_idx0]

        major_full = build_major_fullname([major_name, direction, remark] + extra_vals)
        if col_full is not None:
            ws.cell(r, column=col_full).value = major_full

        if not prov_val or not major_full:
            row_cache[r] = ([], [], None)
            continue

        entries = prov_rules.get(prov_val)
        if not entries:
            row_cache[r] = ([], [], None)
            continue

        text = norm_text_for_match(major_full)
        gongfei_neg_ctx = bool(NEG_CONTEXT_GONGFEI.search(text))
        hukou_value = extract_hukou_shortest(text)

        # 户籍兜底：正则没提出来，但文本明确说有户籍条件
        if not hukou_value and any(p.search(text) for p in HUKOU_FALLBACK_PATTERNS):
            hukou_value = "有户籍要求"

        matched_labels = []
        hit_keys = []
        label_seen = set()

        for entry in entries:
            label = entry["label"]
            if label in label_seen:
                continue

            first_hit = None
            for item in entry["items"]:
                ok, hit_str = match_item(text, item)
                if ok:
                    first_hit = hit_str
                    break

            if first_hit and gongfei_neg_ctx and ("公费" in str(label)):
                continue

            if first_hit:
                matched_labels.append(label)
                hit_keys.append(first_hit)
                label_seen.add(label)

        matched_labels, hit_keys = resolve_label_conflicts(prov_val, matched_labels, hit_keys)
        matched_labels, hit_keys = remove_negative_zhongwai_label(remark, matched_labels, hit_keys)
        row_cache[r] = (matched_labels, hit_keys, hukou_value)

    # 先插入/确保“户籍”列位置，避免 insert_cols 后导致后面已缓存的列号失效
    col_hukou = ensure_col_after(ws, h, OUT_COL_HUKOU, HUKOU_INSERT_AFTER_COL)
    # 插列后再获取其他写回列的最新列号，防止“专业性质”等写到错误列
    col_nature = ensure_col(ws, h, OUT_COL_MAJOR_NATURE)
    col_hits = ensure_col(ws, h, OUT_COL_HITS) if OUTPUT_HITS_COLUMN else None

    for r in range(2, ws.max_row + 1):
        matched_labels, hit_keys, hukou_value = row_cache[r]
        ws.cell(r, column=col_nature).value = (";".join(matched_labels) if matched_labels else None)

        if OUTPUT_HITS_COLUMN and col_hits is not None:
            ws.cell(r, column=col_hits).value = (" ".join(hit_keys) if hit_keys else None)

        if UNIFY_NONEMPTY_HUKOU_TO_FLAG:
            ws.cell(r, column=col_hukou).value = "1" if hukou_value else "0"
        else:
            ws.cell(r, column=col_hukou).value = hukou_value

    return wb


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    rule_wb = load_workbook(RULE_PATH)
    rule_ws = rule_wb.active
    prov_rules, all_labels, label_keycount = build_rules(rule_ws)

    if os.path.isdir(INPUT_PATH):
        targets = [os.path.join(INPUT_PATH, fn) for fn in os.listdir(INPUT_PATH)]
    else:
        targets = [INPUT_PATH]

    for src in targets:
        fn = os.path.basename(src)
        if fn.startswith("~$"):
            continue
        if not (fn.lower().endswith(".xlsx") or fn.lower().endswith(".xlsm")):
            continue

        base, _ = os.path.splitext(fn)
        dst = os.path.join(OUTPUT_DIR, f"{base}_final.xlsx")

        try:
            wb_out = process_one_file(src, prov_rules, all_labels, label_keycount)
            wb_out.save(dst)
            print(f"[OK] {fn} -> {os.path.basename(dst)}")
        except Exception as e:
            print(f"[FAIL] {fn}: {e}")


if __name__ == "__main__":
    main()

[OK] 北京上传文件_04250.xlsx -> 北京上传文件_04250_final.xlsx
[OK] 安徽上传文件_0425.xlsx -> 安徽上传文件_0425_final.xlsx
[OK] 湖北2025年4月27.xlsx -> 湖北2025年4月27_final.xlsx
[OK] 云南2025年4月27.xlsx -> 云南2025年4月27_final.xlsx
[OK] 黑龙江2025年4月27.xlsx -> 黑龙江2025年4月27_final.xlsx
[OK] 吉林2025年4月27.xlsx -> 吉林2025年4月27_final.xlsx
[OK] 辽宁2025年4月27.xlsx -> 辽宁2025年4月27_final.xlsx
[FAIL] .~内蒙古上传文件_0425.xlsx: File is not a zip file
[OK] 四川2025年4月27.xlsx -> 四川2025年4月27_final.xlsx
[OK] 甘肃2025年4月27.xlsx -> 甘肃2025年4月27_final.xlsx
[OK] 宁夏上传文件_0425.xlsx -> 宁夏上传文件_0425_final.xlsx
[OK] 湖南2025年4月27.xlsx -> 湖南2025年4月27_final.xlsx
[OK] 浙江上传文件_0425.xlsx -> 浙江上传文件_0425_final.xlsx
[FAIL] .~福建上传文件_0425.xlsx: File is not a zip file
[OK] 广东上传文件_0425.xlsx -> 广东上传文件_0425_final.xlsx
[OK] 内蒙古上传文件_0425.xlsx -> 内蒙古上传文件_0425_final.xlsx
[OK] 贵州2025年4月27.xlsx -> 贵州2025年4月27_final.xlsx
[OK] 新疆2025年4月27.xlsx -> 新疆2025年4月27_final.xlsx
[OK] 重庆上传文件_0425.xlsx -> 重庆上传文件_0425_final.xlsx
[OK] 福建上传文件_0425.xlsx -> 福建上传文件_0425_final.xlsx
[OK] 青海2025年4月27.xlsx -> 青海20